# CRM Access Governance & Customer Data Protection
## Stage 3 — Governance Rules

This notebook translates the findings from the EDA and Explanatory Analysis into a **transparent, documented, and auditable governance framework**.

The objective is not to reproduce the synthetic dataset rules blindly. Instead, the notebook separates:

1. **Authorization rules** — what a role is allowed to do.
2. **Data sensitivity controls** — how access should change depending on the sensitivity of the data.
3. **Contextual risk rules** — how device, failed logins, anomaly score, access time, and governance indicators affect the decision.
4. **Decision precedence** — which rules take priority when multiple conditions apply.
5. **Governance evidence** — which analytical finding motivated each proposed rule.

> **Important:** these rules are proposed controls for a simulated CRM environment. They are not legal advice, not production security policy, and not direct requirements of LGPD/GDPR.


In [ ]:
# 1. Libraries and Settings

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA_PATH = Path("Permission_Aware_CRM_Governance_Synthetic_50000.csv")

print("Environment ready.")


# 2. Data Loading

The original synthetic dataset is loaded again so the governance framework can be tested against the observed access records.


In [ ]:
df = pd.read_csv(DATA_PATH)

df_rules = df.copy()

df_rules["Is_Blocked"] = (
    df_rules["Access_Decision"]
    .eq("Block")
    .astype(int)
)

print(f"Rows: {df_rules.shape[0]:,}")
print(f"Columns: {df_rules.shape[1]}")
display(df_rules.head())


# 3. Governance Design Principles

The proposed framework follows a layered logic.

### Layer 1 — Authorization
Does the user's role have a legitimate need to perform the requested CRM action?

### Layer 2 — Data Sensitivity
How sensitive is the information involved?

### Layer 3 — Contextual Risk
Does the access context introduce additional risk?

Examples:
- BYOD device;
- repeated failed logins;
- high anomaly score;
- edge-hour access;
- low compliance;
- high-risk CRM action.

### Layer 4 — Final Decision
The framework produces one of three proposed decisions:

- `ALLOW`
- `REVIEW`
- `BLOCK`

The original synthetic dataset only contains `Block` and `Review`. `ALLOW` is introduced here as part of the proposed governance design.


# 4. Proposed Data Sensitivity Classification

The original dataset contains `Data_Sensitivity` levels from 1 to 5.

For governance purposes, we map those levels into descriptive categories.


In [ ]:
sensitivity_policy = pd.DataFrame({
    "Data_Sensitivity": [1, 2, 3, 4, 5],
    "Sensitivity_Label": [
        "Low",
        "Internal",
        "Confidential",
        "Personal / Restricted",
        "Critical / Highly Restricted"
    ],
    "Default_Control": [
        "Standard access",
        "Standard access with logging",
        "Role-based restriction",
        "Enhanced contextual review",
        "Strict restriction and enhanced review"
    ]
})

display(sensitivity_policy)


## Governance Note

The labels above are **internal governance labels created for this project**.

They should not be interpreted as official LGPD classifications. The goal is to create a practical internal control framework that can later be mapped to legal or compliance requirements.


# 5. Proposed Role × CRM Action Access Matrix

This matrix defines the **baseline authorization level** for each role-action combination.

The allowed values are:

- `ALLOW`
- `REVIEW`
- `BLOCK`

The matrix is intentionally explicit so it can be reviewed, audited, and changed without modifying model code.


In [ ]:
roles = [
    "Admin",
    "Manager",
    "Sales Rep",
    "Support",
    "Analyst"
]

actions = [
    "ViewLead",
    "EditLead",
    "CreateOpportunity",
    "ApproveDiscount",
    "ExportCRM",
    "DeleteLead"
]

access_matrix = pd.DataFrame(
    index=roles,
    columns=actions
)

access_matrix.loc["Admin"] = [
    "ALLOW", "ALLOW", "ALLOW",
    "ALLOW", "ALLOW", "ALLOW"
]

access_matrix.loc["Manager"] = [
    "ALLOW", "ALLOW", "ALLOW",
    "ALLOW", "REVIEW", "REVIEW"
]

access_matrix.loc["Sales Rep"] = [
    "ALLOW", "ALLOW", "ALLOW",
    "REVIEW", "REVIEW", "BLOCK"
]

access_matrix.loc["Support"] = [
    "ALLOW", "REVIEW", "BLOCK",
    "BLOCK", "BLOCK", "BLOCK"
]

access_matrix.loc["Analyst"] = [
    "ALLOW", "BLOCK", "BLOCK",
    "BLOCK", "REVIEW", "BLOCK"
]

display(access_matrix)


## Interpretation

The matrix is a **proposed RBAC baseline**, not a rule discovered directly from the dataset.

It should be interpreted as a business-governance design:

- roles should receive only the actions necessary for their function;
- destructive or bulk-access actions should be more restricted;
- some actions should require additional review rather than being automatically allowed.

This separation between **analytical evidence** and **proposed policy design** is intentional.


# 6. Convert Access Matrix to Long Format

A long-form table is easier to document, export, join, and reuse in Power BI.


In [ ]:
access_matrix_long = (
    access_matrix
    .reset_index()
    .rename(columns={"index": "Role"})
    .melt(
        id_vars="Role",
        var_name="CRM_Action",
        value_name="Baseline_Authorization"
    )
)

display(access_matrix_long.head(20))


# 7. Contextual Risk Flags

Contextual rules do not replace authorization.

They operate **after** baseline authorization has been established.


In [ ]:
df_rules["High_Sensitivity_Flag"] = (
    df_rules["Data_Sensitivity"] >= 4
)

df_rules["BYOD_Flag"] = (
    df_rules["Device_Type"] == "BYOD"
)

df_rules["Failed_Login_Flag"] = (
    df_rules["Failed_Logins"] > 0
)

anomaly_threshold = df_rules["Anomaly_Score"].quantile(0.90)

df_rules["High_Anomaly_Flag"] = (
    df_rules["Anomaly_Score"] >= anomaly_threshold
)

min_hour = df_rules["Access_Hour"].min()
max_hour = df_rules["Access_Hour"].max()

df_rules["Edge_Hour_Flag"] = (
    (df_rules["Access_Hour"] <= min_hour + 1) |
    (df_rules["Access_Hour"] >= max_hour - 1)
)

df_rules["Low_Compliance_Flag"] = (
    df_rules["Policy_Compliance_Score"] <
    df_rules["Policy_Compliance_Score"].quantile(0.25)
)

df_rules["Low_Governance_Flag"] = (
    df_rules["Governance_Score"] <
    df_rules["Governance_Score"].quantile(0.25)
)

print(f"High anomaly threshold (90th percentile): {anomaly_threshold:.3f}")


# 8. Contextual Risk Score

A simple transparent score is created for governance testing.

This is intentionally rule-based rather than machine-learning based.

Each condition contributes one point:

- high sensitivity;
- BYOD;
- failed login;
- high anomaly;
- edge-hour access;
- low policy compliance;
- low governance score.

The score is designed for interpretability, not predictive optimization.


In [ ]:
contextual_flags = [
    "High_Sensitivity_Flag",
    "BYOD_Flag",
    "Failed_Login_Flag",
    "High_Anomaly_Flag",
    "Edge_Hour_Flag",
    "Low_Compliance_Flag",
    "Low_Governance_Flag"
]

df_rules["Contextual_Risk_Score"] = (
    df_rules[contextual_flags]
    .sum(axis=1)
)

risk_level_map = {
    0: "LOW",
    1: "LOW",
    2: "MEDIUM",
    3: "MEDIUM",
    4: "HIGH",
    5: "HIGH",
    6: "CRITICAL",
    7: "CRITICAL"
}

df_rules["Contextual_Risk_Level"] = (
    df_rules["Contextual_Risk_Score"]
    .map(risk_level_map)
)

display(
    df_rules[
        contextual_flags +
        ["Contextual_Risk_Score", "Contextual_Risk_Level"]
    ].head()
)


# 9. Rule Catalog

Each governance rule receives:

- a stable Rule ID;
- a description;
- a category;
- a priority;
- a proposed outcome;
- a rationale;
- an evidence source.

This makes the framework auditable and easier to maintain.


In [ ]:
rule_catalog = pd.DataFrame([
    {
        "Rule_ID": "AUTH-001",
        "Category": "Authorization",
        "Priority": 1,
        "Condition": "Permission_Granted = False",
        "Proposed_Decision": "BLOCK",
        "Rationale": "No explicit permission is available for the requested access.",
        "Evidence_Source": "Explanatory Analysis — permission rule check"
    },
    {
        "Rule_ID": "AUTH-002",
        "Category": "Authorization",
        "Priority": 2,
        "Condition": "Role × CRM_Action baseline = BLOCK",
        "Proposed_Decision": "BLOCK",
        "Rationale": "Requested action is outside the proposed role authorization baseline.",
        "Evidence_Source": "Governance design — RBAC matrix"
    },
    {
        "Rule_ID": "AUTH-003",
        "Category": "Authorization",
        "Priority": 3,
        "Condition": "Role × CRM_Action baseline = REVIEW",
        "Proposed_Decision": "REVIEW",
        "Rationale": "Action requires additional approval or contextual review.",
        "Evidence_Source": "Governance design — RBAC matrix"
    },
    {
        "Rule_ID": "CTX-001",
        "Category": "Contextual Risk",
        "Priority": 4,
        "Condition": "Contextual_Risk_Level = CRITICAL",
        "Proposed_Decision": "BLOCK",
        "Rationale": "Multiple concurrent risk signals indicate a critical access context.",
        "Evidence_Source": "EDA + Explanatory Analysis — combined risk factors"
    },
    {
        "Rule_ID": "CTX-002",
        "Category": "Contextual Risk",
        "Priority": 5,
        "Condition": "High sensitivity AND BYOD AND high anomaly",
        "Proposed_Decision": "BLOCK",
        "Rationale": "Highly sensitive access from unmanaged device with anomalous behavior.",
        "Evidence_Source": "Interaction analysis"
    },
    {
        "Rule_ID": "CTX-003",
        "Category": "Contextual Risk",
        "Priority": 6,
        "Condition": "Contextual_Risk_Level = HIGH",
        "Proposed_Decision": "REVIEW",
        "Rationale": "High contextual risk requires manual or secondary validation.",
        "Evidence_Source": "Combined risk factor analysis"
    },
    {
        "Rule_ID": "CTX-004",
        "Category": "Contextual Risk",
        "Priority": 7,
        "Condition": "Data_Sensitivity >= 4 AND Failed_Logins > 0",
        "Proposed_Decision": "REVIEW",
        "Rationale": "Sensitive-data access combined with authentication friction requires review.",
        "Evidence_Source": "Sensitivity and failed-login analysis"
    },
    {
        "Rule_ID": "CTX-005",
        "Category": "Contextual Risk",
        "Priority": 8,
        "Condition": "Contextual_Risk_Level = MEDIUM",
        "Proposed_Decision": "REVIEW",
        "Rationale": "Moderate contextual risk requires additional attention.",
        "Evidence_Source": "Combined risk factor analysis"
    },
    {
        "Rule_ID": "DEFAULT-001",
        "Category": "Default",
        "Priority": 99,
        "Condition": "No higher-priority rule triggered",
        "Proposed_Decision": "ALLOW",
        "Rationale": "Authorized low-risk access follows the default allow path.",
        "Evidence_Source": "Governance design"
    }
])

display(rule_catalog)


# 10. Join Baseline Authorization to Access Records

Each access record receives the proposed role-action baseline from the access matrix.


In [ ]:
df_rules = df_rules.merge(
    access_matrix_long,
    on=["Role", "CRM_Action"],
    how="left"
)

print(
    "Records without baseline authorization mapping:",
    df_rules["Baseline_Authorization"].isna().sum()
)

display(
    df_rules[
        [
            "Role",
            "CRM_Action",
            "Baseline_Authorization"
        ]
    ].head(10)
)


# 11. Governance Decision Engine

The rules are applied in a clear priority order.

### Proposed precedence

1. No permission → `BLOCK`
2. Role-action baseline block → `BLOCK`
3. Critical contextual risk → `BLOCK`
4. Specific high-risk combination → `BLOCK`
5. Baseline review → `REVIEW`
6. High contextual risk → `REVIEW`
7. Sensitive data + failed login → `REVIEW`
8. Medium contextual risk → `REVIEW`
9. Otherwise → `ALLOW`

This ordering is designed to make the rule system deterministic and auditable.


In [ ]:
def governance_decision(row):
    # Rule 1 — explicit permission
    if row["Permission_Granted"] == False:
        return "BLOCK", "AUTH-001"

    # Rule 2 — role-action baseline
    if row["Baseline_Authorization"] == "BLOCK":
        return "BLOCK", "AUTH-002"

    # Rule 3 — critical contextual risk
    if row["Contextual_Risk_Level"] == "CRITICAL":
        return "BLOCK", "CTX-001"

    # Rule 4 — highly sensitive anomalous BYOD access
    if (
        row["High_Sensitivity_Flag"]
        and row["BYOD_Flag"]
        and row["High_Anomaly_Flag"]
    ):
        return "BLOCK", "CTX-002"

    # Rule 5 — baseline requires review
    if row["Baseline_Authorization"] == "REVIEW":
        return "REVIEW", "AUTH-003"

    # Rule 6 — high contextual risk
    if row["Contextual_Risk_Level"] == "HIGH":
        return "REVIEW", "CTX-003"

    # Rule 7 — sensitive data with failed authentication attempts
    if (
        row["High_Sensitivity_Flag"]
        and row["Failed_Login_Flag"]
    ):
        return "REVIEW", "CTX-004"

    # Rule 8 — medium contextual risk
    if row["Contextual_Risk_Level"] == "MEDIUM":
        return "REVIEW", "CTX-005"

    # Default rule
    return "ALLOW", "DEFAULT-001"


In [ ]:
decision_output = df_rules.apply(
    governance_decision,
    axis=1,
    result_type="expand"
)

decision_output.columns = [
    "Proposed_Access_Decision",
    "Triggered_Rule_ID"
]

df_rules = pd.concat(
    [df_rules, decision_output],
    axis=1
)

display(
    df_rules[
        [
            "User_ID",
            "Role",
            "CRM_Action",
            "Permission_Granted",
            "Baseline_Authorization",
            "Contextual_Risk_Level",
            "Proposed_Access_Decision",
            "Triggered_Rule_ID"
        ]
    ].head(20)
)


# 12. Proposed Decision Distribution

The proposed framework contains three outcomes, unlike the original dataset.


In [ ]:
proposed_distribution = (
    df_rules["Proposed_Access_Decision"]
    .value_counts()
    .to_frame("Count")
    .assign(
        Percentage=lambda x:
        x["Count"] / len(df_rules) * 100
    )
)

display(proposed_distribution.round(2))


# 13. Rule Trigger Frequency

This shows which rules dominate the proposed decision framework.


In [ ]:
rule_trigger_summary = (
    df_rules["Triggered_Rule_ID"]
    .value_counts()
    .to_frame("Records")
    .reset_index()
    .rename(columns={"index": "Triggered_Rule_ID"})
)

rule_trigger_summary["Percentage"] = (
    rule_trigger_summary["Records"] /
    len(df_rules) * 100
)

rule_trigger_summary = rule_trigger_summary.merge(
    rule_catalog[
        [
            "Rule_ID",
            "Category",
            "Proposed_Decision",
            "Rationale"
        ]
    ],
    left_on="Triggered_Rule_ID",
    right_on="Rule_ID",
    how="left"
)

display(rule_trigger_summary.round(2))


# 14. Comparison with the Synthetic Access Decision

The purpose of this comparison is **not** to maximize agreement.

The original target only contains `Block` and `Review`, while the proposed framework also includes `ALLOW`.

Instead, the comparison helps identify:

- where the proposed framework is stricter;
- where it is more permissive;
- where additional business validation would be required.


In [ ]:
comparison = pd.crosstab(
    df_rules["Access_Decision"],
    df_rules["Proposed_Access_Decision"],
    margins=True
)

display(comparison)


# 15. Agreement on Block Decisions

Because both systems contain a `BLOCK` outcome, block agreement can still be examined separately.


In [ ]:
df_rules["Original_Block"] = (
    df_rules["Access_Decision"] == "Block"
)

df_rules["Proposed_Block"] = (
    df_rules["Proposed_Access_Decision"] == "BLOCK"
)

block_comparison = pd.crosstab(
    df_rules["Original_Block"],
    df_rules["Proposed_Block"],
    margins=True
)

display(block_comparison)


# 16. Governance Exceptions

Records where the proposed framework and the synthetic decision differ are useful for review.

These cases should be treated as **policy exceptions / validation candidates**, not automatically as errors.


In [ ]:
exceptions = df_rules[
    (
        (df_rules["Access_Decision"] == "Block") &
        (df_rules["Proposed_Access_Decision"] != "BLOCK")
    )
    |
    (
        (df_rules["Access_Decision"] == "Review") &
        (df_rules["Proposed_Access_Decision"] == "BLOCK")
    )
].copy()

print(f"Potential governance exceptions: {len(exceptions):,}")

display(
    exceptions[
        [
            "User_ID",
            "Role",
            "CRM_Action",
            "Device_Type",
            "Data_Sensitivity",
            "Failed_Logins",
            "Anomaly_Score",
            "Policy_Compliance_Score",
            "Governance_Score",
            "Permission_Granted",
            "Baseline_Authorization",
            "Contextual_Risk_Level",
            "Access_Decision",
            "Proposed_Access_Decision",
            "Triggered_Rule_ID"
        ]
    ].head(30)
)


# 17. Rule-Level Validation Summary

This table compares the original block rate within each rule-triggered group.

It helps evaluate whether the proposed rule is aligned with patterns observed in the synthetic dataset.


In [ ]:
rule_validation = (
    df_rules.groupby("Triggered_Rule_ID")
    .agg(
        Records=("User_ID", "size"),
        Original_Block_Rate=("Is_Blocked", "mean")
    )
    .reset_index()
)

rule_validation["Original_Block_Rate"] *= 100

rule_validation = rule_validation.merge(
    rule_catalog[
        [
            "Rule_ID",
            "Category",
            "Proposed_Decision",
            "Rationale"
        ]
    ],
    left_on="Triggered_Rule_ID",
    right_on="Rule_ID",
    how="left"
)

display(
    rule_validation
    .sort_values("Original_Block_Rate", ascending=False)
    .round(2)
)


# 18. Governance Policy Table

This table consolidates the framework into a format suitable for documentation or export.


In [ ]:
governance_policy_table = rule_catalog[
    [
        "Rule_ID",
        "Category",
        "Priority",
        "Condition",
        "Proposed_Decision",
        "Rationale",
        "Evidence_Source"
    ]
].sort_values("Priority")

display(governance_policy_table)


# 19. Governance Control Architecture

The proposed logic can be summarized as:

```text
Access Request
      |
      v
Explicit Permission?
      |
  +---+---+
  |       |
 NO      YES
  |       |
BLOCK     v
     Role × Action Matrix
             |
       +-----+-----+
       |           |
     BLOCK      ALLOW/REVIEW
       |           |
       v           v
     BLOCK   Contextual Risk
                   |
             +-----+-----+
             |     |     |
            LOW  MEDIUM  HIGH/CRITICAL
             |     |       |
          ALLOW  REVIEW  REVIEW/BLOCK
```

This structure separates **authorization** from **contextual risk**, which makes the final decision easier to explain and audit.


# 20. Governance Documentation Template

Every production rule should eventually include:

- Rule ID
- Rule name
- Business owner
- Description
- Trigger condition
- Decision outcome
- Priority
- Exception process
- Evidence / justification
- Data fields used
- Review frequency
- Effective date
- Last review date
- Change history
- Approval status

The current notebook implements only the analytical prototype of this structure.


# 21. Findings to Document

Complete this section after reviewing the outputs.

## 21.1 Authorization
- Which roles have the broadest proposed access?
- Which CRM actions are most restricted?
- Which role-action combinations require review?

## 21.2 Contextual Risk
- Which contextual factors trigger the most rules?
- How often does high or critical risk occur?
- Which combinations deserve stricter review?

## 21.3 Rule Validation
- Which proposed rules align most strongly with original block decisions?
- Which rules appear overly strict?
- Which rules appear too permissive?

## 21.4 Exceptions
- What are the most common disagreement patterns?
- Do disagreements concentrate in specific roles or actions?
- Which rules should be reconsidered?

## 21.5 Governance Conclusions
1.
2.
3.


# 22. Limitations

1. The access matrix is a **proposed governance design**, not an observed business policy.
2. The dataset is synthetic.
3. The dataset does not contain actual customer-level PII.
4. `ALLOW` does not exist in the original `Access_Decision` target.
5. Contextual thresholds are exploratory and data-driven.
6. The current risk score assigns equal weights for interpretability.
7. A real implementation would require validation by security, legal, compliance, business owners, and system administrators.
8. Legal compliance cannot be inferred solely from these analytical rules.


# 23. Next Step — Analytical Data Model

The next stage is:

## Stage 4 — Analytical Data Model

Planned outputs:

- fact table for CRM access events;
- role dimension;
- action dimension;
- device dimension;
- sensitivity dimension;
- rule dimension;
- date/time dimension;
- governance decision fields;
- risk metrics;
- star-schema design;
- Power BI-ready analytical layer.

The goal will be to convert the governance framework into a model that supports monitoring, reporting, and decision-making.
